# B5: is the safety gain bought with refusals? (local CPU)
Harmful prompts only (all severities, same as the unsafe metric), 0.6B judge, `paired_ok` seeds.

P(unsafe) = P(not refused) × P(unsafe | not refused). For each adapter vs base, the change in unsafe rate is split (Shapley, two factors) into
**refusing more** (change in P(not refused)) and **answering more safely** (change in P(unsafe | not refused)).
Answers are loaded with the exact code of `B4_analysis` (same sources, reuse checks and prompt set).

In [ ]:
from pathlib import Path
src = Path('nbsrc/B4_analysis.py').read_text()
cut = src.index('MODELS = sorted(ROWS)')
head = src[:cut]
head = head.replace('N_BOOT = 10000', 'N_BOOT = 0')
exec(compile(head, 'B4_analysis_head', 'exec'))
print(len(E), 'prompts;', sorted(ROWS))

In [ ]:

import numpy as np, pandas as pd
H = E[E.kind == "harmful"].reset_index(drop=True)
def lab(m, j="g06"):
    d = ROWS[m]
    u = np.array([{"Unsafe": 1.0, "Safe": 0.0}.get(d[p][f"{j}_label"], np.nan) if p in d else np.nan for p in H.prompt_id])
    r = np.array([np.nan if (p not in d or d[p][f"{j}_refused"] is None) else float(d[p][f"{j}_refused"]) for p in H.prompt_id])
    return u, r
def stats(m, mask, j="g06"):
    u, r = lab(m, j)
    k = mask & ~np.isnan(u) & ~np.isnan(r)
    u, r = u[k], r[k]
    A = (r == 0).mean()                       # P(not refused)
    B = u[r == 0].mean() if (r == 0).any() else 0.0   # P(unsafe | not refused)
    return {"n": int(k.sum()), "unsafe": u.mean(), "refused": 1 - A, "not_refused": A, "unsafe_given_answered": B,
            "unsafe_given_refused": u[r == 1].mean() if (r == 1).any() else np.nan}
GROUPS = {"base": ["base"], "en": ["en_s0", "en_s1", "en_s2"], "ja": ["ja_s0", "ja_s1", "ja_s2"],
          "bi": ["bi_s0", "bi_s1", "bi_s2"], "en2": ["en2_s0", "en2_s1", "en2_s2"]}
SLICES = {"English prompts": (H.lang == "en").values, "Japanese prompts": (H.lang == "ja").values, "All prompts": np.ones(len(H), bool)}
rows = []
for sl, mask in SLICES.items():
    b = stats("base", mask)
    for g, ms in GROUPS.items():
        per = []
        for m in ms:
            s = stats(m, mask)
            dA, dB = s["not_refused"] - b["not_refused"], s["unsafe_given_answered"] - b["unsafe_given_answered"]
            ref_part = dA * (b["unsafe_given_answered"] + s["unsafe_given_answered"]) / 2   # from refusing more/less
            ans_part = dB * (b["not_refused"] + s["not_refused"]) / 2                       # from safer/less safe answers
            per.append({**s, "d_unsafe": s["unsafe"] - b["unsafe"], "d_from_refusing": ref_part, "d_from_answers": ans_part})
        P = pd.DataFrame(per).mean()
        tot = P.d_from_refusing + P.d_from_answers
        rows.append({"slice": sl, "model": g, "unsafe": P.unsafe, "refused_harmful": P.refused,
                     "unsafe_given_answered": P.unsafe_given_answered, "unsafe_given_refused": P.unsafe_given_refused,
                     "d_unsafe": P.d_unsafe, "d_from_refusing": P.d_from_refusing, "d_from_answers": P.d_from_answers,
                     "refusal_share": (P.d_from_refusing / tot) if g != "base" and abs(tot) > 1e-9 else np.nan, "n": int(P.n)})
R = pd.DataFrame(rows)
pd.set_option("display.width", 250)
fmt = R.copy()
for col in ["unsafe", "refused_harmful", "unsafe_given_answered", "unsafe_given_refused"]:
    fmt[col] = (100 * fmt[col]).round(1)
for col in ["d_unsafe", "d_from_refusing", "d_from_answers"]:
    fmt[col] = (100 * fmt[col]).round(2)
fmt["refusal_share"] = (100 * fmt["refusal_share"]).round(0)
print(fmt.to_string(index=False))


In [ ]:

# By culture x language as well (for the culture question).
rows2 = []
for cu in CULT:
    for lg in LANGS:
        mask = ((H.culture == cu) & (H.lang == lg)).values
        b = stats("base", mask)
        for g, ms in GROUPS.items():
            if g == "base": continue
            per = []
            for m in ms:
                s = stats(m, mask)
                dA, dB = s["not_refused"] - b["not_refused"], s["unsafe_given_answered"] - b["unsafe_given_answered"]
                per.append({"unsafe": s["unsafe"], "refused": s["refused"], "ucond": s["unsafe_given_answered"],
                            "ref": dA * (b["unsafe_given_answered"] + s["unsafe_given_answered"]) / 2,
                            "ans": dB * (b["not_refused"] + s["not_refused"]) / 2})
            P = pd.DataFrame(per).mean()
            rows2.append({"culture": cu, "lang": lg, "model": g, "base_unsafe": round(100*b["unsafe"],1), "unsafe": round(100*P.unsafe,1),
                          "base_refused": round(100*b["refused"],1), "refused": round(100*P.refused,1),
                          "base_ucond": round(100*b["unsafe_given_answered"],1), "ucond": round(100*P.ucond,1),
                          "from_refusing": round(100*P.ref,2), "from_answers": round(100*P.ans,2),
                          "refusal_share": round(100*P.ref/(P.ref+P.ans)) if abs(P.ref+P.ans) > 1e-9 else None})
print(pd.DataFrame(rows2).to_string(index=False))


In [ ]:

# Raw counts pooled over training runs (k/n summed across s0..s2): unsafe, refused, unsafe among answered.
def counts(m, mask, j="g06"):
    u, r = lab(m, j)
    k = mask & ~np.isnan(u) & ~np.isnan(r)
    u, r = u[k], r[k]
    return int((u == 1).sum()), int(k.sum()), int((r == 1).sum()), int(((u == 1) & (r == 0)).sum()), int((r == 0).sum())
out = []
cells = {"English": (H.lang == "en").values, "Japanese": (H.lang == "ja").values, "All": np.ones(len(H), bool)}
for cu in CULT:
    for lg in LANGS:
        cells[f"{cu}/{lg}"] = ((H.culture == cu) & (H.lang == lg)).values
for name, mask in cells.items():
    for g, ms in GROUPS.items():
        t = np.sum([counts(m, mask) for m in ms], axis=0)
        out.append({"slice": name, "model": g, "unsafe": f"{t[0]}/{t[1]}", "refused": f"{t[2]}/{t[1]}", "unsafe_when_answered": f"{t[3]}/{t[4]}"})
print(pd.DataFrame(out).to_string(index=False))
